# 🛡️ SkyGuard: Unsupervised Anomaly Detection for Weather Station Sensors

## Real-Time Fault Diagnosis Across IoT Meteorological Networks

Automated Weather Stations (AWS) operate autonomously in remote environments, monitoring critical atmospheric conditions including **temperature, relative humidity, barometric pressure, wind speed, wind direction, rainfall, and solar radiation** sampled at 10-minute intervals.

Because ground-truth anomaly labels do not exist in real-world meteorological deployments, this system uses an **unsupervised machine learning and physical guardrail architecture** to detect:
1. **Impulse Spikes**: Transient hardware glitches, electrostatic discharge, or bit flips.
2. **Stuck-at Faults (Flatlines)**: Mechanical freezing of anemometers or frozen digital serial bus (I2C/SPI).
3. **Calibration Drift**: Progressive baseline degradation in pressure sensors or pyranometers.
4. **Dropouts**: Sudden zero-reading drops or missing data bursts.
5. **Multivariate Physical Contradictions**: Physically impossible atmospheric states (e.g., solar irradiance at midnight, relative humidity $> 100\%$, or temperature plunging below dew point).

---

### Key Requirements Demonstrated in this Notebook:
1. **Data Pipeline**: Ingestion of CSV/Parquet time series, temporal resampling, gap interpolation, and missing timestamp handling.
2. **Feature Engineering**: Rolling statistics (mean, std, spread), rates of change (differencing & acceleration), thermodynamic couplings (Magnus-Tetens dew point, Vapor Pressure Deficit), and cyclical diurnal/seasonal encodings.
3. **Four Unsupervised Detectors**:
   - **Isolation Forest**: High-dimensional recursive random partitioning.
   - **1D Convolutional Autoencoder**: Temporal reconstruction error ($MSE$).
   - **Statistical Baseline**: Rolling Z-score and Hampel filter (rolling median & MAD).
   - **Local Outlier Factor (LOF)**: Local density deviation estimation.
4. **Dynamic Thresholding**: Percentile-based and MAD-based thresholding with a continuous sensitivity parameter $S \in (0, 1]$.
5. **Root Cause Categorization**: Diagnostic classification into `spike`, `stuck`, `drift`, `dropout`, and `multivariate`.
6. **Ground-Truth Benchmark Evaluation**: Precision, Recall, F1-Score, and ROC/PR AUC curves on injected sensor faults.


## 1. Environment Setup & Pipeline Imports

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is on Python path
sys.path.insert(0, os.path.abspath(".."))

from src.config import load_config
from src.data.loader import DataLoader
from src.data.synthetic import WeatherAnomalyGenerator
from src.features.engineer import FeatureEngineer
from src.detector import SkyGuardDetector
from src.evaluation import ModelEvaluator

# Plot styling
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["font.size"] = 10

config = load_config("../configs/config.yaml")
print(f"Loaded configuration for {len(config.sensors)} weather sensors.")


## 2. Atmospheric Simulation & Injected Fault Telemetry

To evaluate unsupervised methods without relying on ground-truth labels during training, we generate realistic multi-station 10-minute sensor telemetry incorporating:
- **Diurnal thermal cycle**: Sinusoidal temperature peaking at 14:30.
- **Solar elevation model**: Solar irradiance bound to day/night zenith angle.
- **Thermodynamic psychrometrics**: Relative humidity inversely coupled to ambient temperature.
- **Synoptic weather trends**: Barometric pressure fronts and atmospheric tides.
- **Episodic precipitation**: Discrete rainfall bursts.

Known sensor faults are injected and tracked in the ground truth columns (`is_anomaly`, `anomaly_type`) solely for post-hoc validation.


In [ ]:
generator = WeatherAnomalyGenerator(config, seed=42)

# Generate 14 days of telemetry for 2 stations with 5% injected fault ratio
telemetry_df = generator.generate(
    stations=["STATION_ALPHA", "STATION_BETA"],
    days=14,
    freq_minutes=10,
    anomaly_ratio=0.05
)

print(f"Generated {len(telemetry_df)} observations across stations.")
print("\nGround-truth anomaly archetype distribution:")
print(telemetry_df["anomaly_type"].value_counts())

telemetry_df.head(6)


## 3. Visualizing Sensor Telemetry & Injected Faults

In [ ]:
st_alpha = telemetry_df[telemetry_df["station_id"] == "STATION_ALPHA"].reset_index(drop=True)
plot_slice = st_alpha.iloc[:432]  # First 3 days (432 timestamps)

fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)

# 1. Temperature
axes[0].plot(plot_slice["timestamp"], plot_slice["temperature"], label="Temperature (°C)", color="#d95f02", lw=1.5)
anom_mask = plot_slice["is_anomaly"] == 1
axes[0].scatter(plot_slice.loc[anom_mask, "timestamp"], plot_slice.loc[anom_mask, "temperature"],
                color="red", s=35, zorder=5, label="Fault Injected")
axes[0].set_ylabel("Temperature (°C)")
axes[0].legend(loc="upper right")
axes[0].set_title("Station Alpha: Multi-Sensor Telemetry (3-Day Excerpt)")

# 2. Humidity & Dew Point
axes[1].plot(plot_slice["timestamp"], plot_slice["humidity"], label="Humidity (%)", color="#1b9e77", lw=1.5)
axes[1].scatter(plot_slice.loc[anom_mask, "timestamp"], plot_slice.loc[anom_mask, "humidity"],
                color="red", s=35, zorder=5)
axes[1].set_ylabel("Humidity (%)")
axes[1].legend(loc="upper right")

# 3. Barometric Pressure
axes[2].plot(plot_slice["timestamp"], plot_slice["barometric_pressure"], label="Pressure (hPa)", color="#7570b3", lw=1.5)
axes[2].scatter(plot_slice.loc[anom_mask, "timestamp"], plot_slice.loc[anom_mask, "barometric_pressure"],
                color="red", s=35, zorder=5)
axes[2].set_ylabel("Pressure (hPa)")
axes[2].legend(loc="upper right")

# 4. Solar Radiation
axes[3].plot(plot_slice["timestamp"], plot_slice["solar_radiation"], label="Solar Radiation (W/m²)", color="#e7298a", lw=1.5)
axes[3].scatter(plot_slice.loc[anom_mask, "timestamp"], plot_slice.loc[anom_mask, "solar_radiation"],
                color="red", s=35, zorder=5)
axes[3].set_ylabel("Solar (W/m²)")
axes[3].set_xlabel("Timestamp")
axes[3].legend(loc="upper right")

plt.tight_layout()
plt.show()


## 4. Meteorological Feature Engineering

To give unsupervised algorithms physical context, the feature pipeline extracts:
1. **Multi-Scale Rolling Statistics**: 1h (6 intervals), 6h (36 intervals), and 24h (144 intervals) rolling mean, standard deviation, spread, and local z-scores.
2. **Temporal Dynamics**: First difference (velocity $\Delta x$) and second difference (acceleration $\Delta^2 x$).
3. **Atmospheric Thermodynamics**:
   - **Magnus-Tetens Dew Point**: $T_{\text{dew}} = \frac{b \cdot \gamma(T, RH)}{a - \gamma(T, RH)}$ where $\gamma = \frac{a T}{b + T} + \ln(RH / 100)$.
   - **Dew Point Depression**: $(T - T_{\text{dew}})$ (must be $\ge 0$ in normal air).
   - **Vapor Pressure Deficit (VPD)**: Atmospheric evaporative drying demand.
4. **Cyclical Temporal Coordinates**: $\sin/\cos$ diurnal (24h) and annual seasonal (365.25d) encodings.


In [ ]:
engineer = FeatureEngineer(config)
engineered_df = engineer.transform(telemetry_df)
feature_cols = engineer.get_feature_column_names(engineered_df)

print(f"Engineered {len(feature_cols)} features from raw sensor data:")
print(f"Features: {feature_cols[:10]} ... (+{len(feature_cols) - 10} more)")

# Display excerpt of engineered thermodynamic and temporal features
engineered_df[["timestamp", "temperature", "dew_point", "dew_point_depression", "vapor_pressure_deficit", "temperature_roc", "hour_sin"]].head(5)


## 5. Unsupervised Model Architecture & Ensemble

We train and evaluate four diverse unsupervised anomaly detection algorithms:

| Model | Underlying Detection Principle | Primary Fault Sensitivity |
| :--- | :--- | :--- |
| **Isolation Forest** | Isolates observations via recursive random hyperplanes in feature space | Point spikes, sudden outliers |
| **Conv1D Autoencoder** | Deep neural temporal reconstruction; computes $MSE = \frac{1}{D} \sum (x - \hat{x})^2$ | Multivariate breakdown, non-linear physical decouple |
| **Statistical Baseline** | Rolling Z-score and Hampel filter (rolling median & $1.4826 \times MAD$) | High-frequency noise, extreme step shocks |
| **Local Outlier Factor (LOF)**| Measures density of sample relative to its $k$-nearest neighbors | Micro-clusters, localized density drops |
| **SkyGuard Ensemble** | Calibrated weighted meta-scoring consensus in $[0, 1]$ | Comprehensive, balanced fault coverage |


In [ ]:
# Fit detector on clean reference baseline
clean_baseline = generator.generate(stations=["STATION_ALPHA"], days=7, anomaly_ratio=0.0)
detector = SkyGuardDetector(config)
detector.fit(clean_baseline)

print("SkyGuard detector trained on clean reference baseline.")

# Predict on test telemetry containing injected anomalies
results_df = detector.predict(st_alpha, sensitivity=0.5, threshold_method="percentile")
print(f"Predicted on {len(results_df)} samples. Dynamic threshold: {results_df['anomaly_threshold'].iloc[0]:.4f}")
print(f"Detected anomalies: {results_df['predicted_anomaly'].sum()} ({results_df['predicted_anomaly'].mean():.2%})")


## 6. Comparing Model Anomaly Scores & Dynamic Thresholding

Below, we visualize the continuous normalized anomaly scores $[0, 1]$ generated by each algorithm, compared against the composite ensemble score and dynamic threshold cutoff.


In [ ]:
sub_res = results_df.iloc[100:400]

plt.figure(figsize=(16, 9))

plt.subplot(3, 1, 1)
plt.plot(sub_res["timestamp"], sub_res["temperature"], label="Temperature (°C)", color="#333333", lw=1.2)
anoms = sub_res[sub_res["predicted_anomaly"] == 1]
plt.scatter(anoms["timestamp"], anoms["temperature"], color="red", s=40, zorder=5, label="Flagged Anomaly")
plt.title("Sensor Signal with Flagged Anomalies")
plt.ylabel("Temperature (°C)")
plt.legend(loc="upper right")

plt.subplot(3, 1, 2)
plt.plot(sub_res["timestamp"], sub_res["score_isolation_forest"], label="Isolation Forest", alpha=0.7, lw=1.2)
plt.plot(sub_res["timestamp"], sub_res["score_autoencoder"], label="Conv1D Autoencoder", alpha=0.7, lw=1.2)
plt.plot(sub_res["timestamp"], sub_res["score_statistical"], label="Statistical (Hampel/Z)", alpha=0.7, lw=1.2)
plt.plot(sub_res["timestamp"], sub_res["score_lof"], label="Local Outlier Factor", alpha=0.7, lw=1.2)
plt.ylabel("Model Score [0, 1]")
plt.title("Individual Unsupervised Model Anomaly Scores")
plt.legend(loc="upper right")

plt.subplot(3, 1, 3)
plt.plot(sub_res["timestamp"], sub_res["anomaly_score"], label="SkyGuard Ensemble Score", color="#0052cc", lw=1.8)
plt.axhline(sub_res["anomaly_threshold"].iloc[0], color="crimson", ls="--", lw=1.5,
            label=f"Dynamic Threshold ({sub_res['anomaly_threshold'].iloc[0]:.3f})")
plt.fill_between(sub_res["timestamp"], sub_res["anomaly_threshold"].iloc[0], 1.0, color="crimson", alpha=0.1)
plt.ylabel("Ensemble Score")
plt.xlabel("Timestamp")
plt.title("Consensus Ensemble Score vs. Dynamic Sensitivity Cutoff")
plt.legend(loc="upper right")

plt.tight_layout()
plt.show()


## 7. How to Tune Sensitivity

The dynamic threshold allows operators to tune the balance between detection rate and false alarms via a single continuous parameter:

$$\text{Sensitivity } S \in (0, 1]$$

- **Low Sensitivity ($S \approx 0.1 - 0.3$)**: High threshold $\tau \approx 0.95 - 0.99$. Strict filter reserved for severe hardware catastrophic dropouts and unphysical spikes. Minimal false alarms.
- **Balanced Sensitivity ($S \approx 0.5$)**: Default threshold $\tau \approx 0.85 - 0.92$. Detects subtle drift and physical decouples while keeping false alarms low.
- **High Sensitivity ($S \approx 0.7 - 1.0$)**: Low threshold $\tau \approx 0.60 - 0.75$. Aggressive surveillance capturing minor calibration drifts and micro-flatlines, suited for critical airport or research deployments.


In [ ]:
sensitivities = np.linspace(0.1, 0.95, 18)
precisions, recalls, f1s, thresholds = [], [], [], []

y_true = st_alpha["is_anomaly"].values
raw_scores = results_df["anomaly_score"].values

for s in sensitivities:
    th = detector.compute_dynamic_threshold(raw_scores, sensitivity=s, method="percentile")
    preds = (raw_scores >= th).astype(int)

    tp = np.sum((preds == 1) & (y_true == 1))
    fp = np.sum((preds == 1) & (y_true == 0))
    fn = np.sum((preds == 0) & (y_true == 1))

    p = tp / (tp + fp) if (tp + fp) > 0 else 0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0
    f = 2 * p * r / (p + r) if (p + r) > 0 else 0

    precisions.append(p)
    recalls.append(r)
    f1s.append(f)
    thresholds.append(th)

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(sensitivities, precisions, marker="o", label="Precision", color="#1b9e77")
plt.plot(sensitivities, recalls, marker="s", label="Recall", color="#d95f02")
plt.plot(sensitivities, f1s, marker="^", label="F1-Score", color="#7570b3", lw=2)
plt.xlabel("Sensitivity Parameter S")
plt.ylabel("Validation Metric")
plt.title("Precision-Recall Trade-off vs. Sensitivity S")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(sensitivities, thresholds, marker="d", color="crimson", lw=2)
plt.xlabel("Sensitivity Parameter S")
plt.ylabel("Dynamic Threshold Value")
plt.title("Dynamic Threshold Cutoff vs. Sensitivity S")

plt.tight_layout()
plt.show()


## 8. Quantitative Model Comparison Benchmark

We evaluate all 4 unsupervised detectors and the SkyGuard Ensemble against ground-truth injected labels, reporting:
- **Precision, Recall, F1-Score**
- **ROC-AUC & PR-AUC (Average Precision)**
- **Inference Latency (ms per sample)**
- **Breakdown by Fault Archetype (`spike`, `stuck`, `drift`, `dropout`, `multivariate`)**


In [ ]:
evaluator = ModelEvaluator(config)
report = evaluator.run_benchmark(days=14, anomaly_ratio=0.05, sensitivity=0.5)
evaluator.print_benchmark_table(report)


## 9. Root Cause Categorization & Diagnosis

When an anomaly score exceeds the dynamic threshold, SkyGuard performs physics-informed diagnostic attribution:
- **Spike**: High rate of change $|\Delta x| > 	ext{limit}$ or local z-score $> 3.8$ that reverts.
- **Stuck**: 0 rolling spread across consecutive intervals on active channels.
- **Drift**: Monotonic cumulative deviation from 24h baseline without step velocity.
- **Dropout**: Sudden plunge to zero or below physical sensor minimums.
- **Multivariate**: Cross-sensor physical rule violations (e.g. night solar, dew point inversion).


In [ ]:
detected_anoms = results_df[results_df["predicted_anomaly"] == 1]
print(f"Total anomalies flagged: {len(detected_anoms)}")
print("\nDiagnosed Anomaly Types:")
print(detected_anoms["predicted_type"].value_counts())

print("\nDiagnosed Impacted Sensors:")
print(detected_anoms["diagnosed_sensor"].value_counts())

# Display sample of diagnosed faults
detected_anoms[["timestamp", "anomaly_score", "predicted_type", "diagnosed_sensor", "temperature", "humidity", "barometric_pressure"]].head(8)


## 10. FastAPI REST API Endpoint Demonstration

SkyGuard provides production REST endpoints returning JSON:
- `POST /detect`: Time-series anomaly detection payload.
- `GET /stations`: Weather station network health overview.
- `GET /anomalies`: Filterable query interface for detected anomalies.


In [ ]:
from fastapi.testclient import TestClient
from src.api import app

client = TestClient(app)

# 1. Health check
health_res = client.get("/health").json()
print("Health Status:", health_res)

# 2. Get Monitored Stations
stations_res = client.get("/stations").json()
print(f"Monitored Stations: {len(stations_res)}")
for st in stations_res:
    print(f"  - {st['station_id']}: Status={st['status']}, Records={st['records_count']}, Anomalies={st['total_anomalies_detected']}")

# 3. Post Detection Payload
sample_payload = {
    "records": [
        {
            "timestamp": "2026-06-01T12:00:00Z",
            "station_id": "STATION_ALPHA",
            "temperature": 24.5,
            "humidity": 65.0,
            "barometric_pressure": 1013.25,
            "wind_speed": 4.2,
            "wind_direction": 180.0,
            "rainfall": 0.0,
            "solar_radiation": 750.0
        },
        {
            "timestamp": "2026-06-01T12:10:00Z",
            "station_id": "STATION_ALPHA",
            "temperature": 48.2,  # Sudden unphysical jump
            "humidity": 15.0,
            "barometric_pressure": 1013.20,
            "wind_speed": 4.5,
            "wind_direction": 182.0,
            "rainfall": 0.0,
            "solar_radiation": 760.0
        }
    ],
    "sensitivity": 0.6,
    "threshold_method": "percentile"
}

detect_res = client.post("/detect", json=sample_payload).json()
print("\nDetect Endpoint Response:")
print(f"  Status: {detect_res['status']}")
print(f"  Threshold: {detect_res['threshold']}")
print(f"  Anomalies Detected: {detect_res['anomalies_detected']}")
for item in detect_res["results"]:
    print(f"    Timestamp: {item['timestamp']} | Score: {item['anomaly_score']} | Flag: {item['predicted_anomaly']} | Type: {item['predicted_type']}")
